# Lab 4: SELF-ASSESSMENT: Profiling Memory Access Patterns in LLMs

**Course**: *Dataflow Architectures for AI: From Principles to Practice*
**Week**: 4, Memory Systems
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research)
**Lab Instructor**: Dr. Rashed Z. Bhatti (IBM Research)
**Runtime**: Coursera Labs (CPU only — **no GPU needed here**)
**Expected time**: 15–20 minutes

---

> **Self-Assessment notebook — the 40 points here are ungraded.** This is where
> you check how well your Lab 4 run went. You do not write code or run
> measurements here — you paste the `lab4_results.json` your Colab run produced,
> and all **40 ungraded Self-Assessment points** are checked from it
> automatically. There is no written analysis and no human review.
>
> **The graded item for this lab is the separate Concept Check quiz** (10 points,
> auto-graded in the Coursera Graded Assignment section). It is mandatory and
> requires **80%** to pass. Nothing in *this* notebook affects your course grade.

## Learning objectives
This notebook assesses the objectives of the Colab lab:
1. Capture per-operator allocation and deallocation with `torch.profiler` (`profile_memory=True`).
2. Identify the memory-intensive operators and the peak allocation during decode.
3. Measure *effective* memory bandwidth against an analytically known byte count, and explain why allocator activity is not bandwidth.
4. Derive the KV-cache size formula, confirm it against measured peak-memory growth, and size it for a scratchpad.
5. Predict where a scratchpad hierarchy (Spyre) helps versus a GPU cache, per bottleneck.

## Prerequisites
A completed run of
[Lab4_Profiling_Memory_Access_STUDENT.ipynb](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/gpu-labs/student/Lab4_Profiling_Memory_Access_STUDENT.ipynb)
on Google Colab (free T4), with its final cell run so that **`lab4_results.json`**
exists.

## Why this notebook exists

Lab 4 profiles memory on a real GPU, so you run it on **Google Colab** (free T4).
This Coursera notebook is where your results get **checked, for your own
feedback**.

The grading environment has **no GPU, no model download, and no network
access**, so it cannot re-run your measurements. What it *can* do — and what it
does — is check that the numbers you measured are **internally consistent**: the
KV byte count has to follow from your own model config, the effective bandwidth
from the bytes moved and the time taken, the per-token growth from a refit of
your own sweep points, the Landscape Map row from the raw measurements behind it.
Those relationships hold for any real run on any GPU, and they break when values
are edited by hand.

## The workflow

| Step | Where | What you do |
|------|-------|-------------|
| 1 | GitHub → Colab | Open [Lab4_Profiling_Memory_Access_STUDENT.ipynb](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/gpu-labs/student/Lab4_Profiling_Memory_Access_STUDENT.ipynb), then **File → Save a copy in Drive** |
| 2 | Colab | Set **Runtime → Change runtime type → T4 GPU**, then complete every TODO and run all cells |
| 3 | Colab | Run the final cell. It writes **`lab4_results.json`** |
| 4 | Colab | Copy the JSON it prints, or open the file browser (📁, left sidebar) and double-click `lab4_results.json` |
| 5 | **Here** | Paste it into the `RESULTS_JSON` cell at the top of this notebook, then **Kernel → Restart & Run All** for your self-assessment |

> **The free Colab T4 is all you need.** Every point below is reachable from a
> single free-tier run. You do not need Colab Pro, and you do not need an L4,
> V100 or A100. If Colab happens to assign you a different GPU, your
> self-assessment is unaffected — Parts A and B are hardware-independent
> arithmetic, and Part C uses deliberately wide bands.
>
> **Students are encouraged to experiment with other GPUs and accelerators to
> expand their learning.** Comparing a T4's measured copy bandwidth against an
> A100's is genuinely instructive. There is **no extra credit** for doing so; all
> 40 Self-Assessment points come from a single free-tier run.

## How this notebook is checked (40 ungraded Self-Assessment points)

| Part | Points | What it checks |
|------|-------:|----------------|
| **A — Structural invariants** | 14 | Facts true of *any* correct run: units, model identity, table shape, required fields |
| **B — Self-consistency** | 18 | Your derived values recomputed from your own raw measurements |
| **C — Physics & plausibility** | 8 | Ordering, wide sanity bands, and the allocator-vs-bandwidth distinction |
| **Ungraded notebook total** | **40** | fully auto-graded from your JSON for Self-Assessment |

**40 points is a full self-assessment score, and a free-tier T4 run earns all of
it.** Every check is **auto-graded from the JSON you paste** — there is no written
analysis and no human review. Parts A and B (32 points) are
hardware-independent; Part C uses wide bands.

A **score table** near the end of this notebook reports your 40-point
self-assessment total, so you know exactly which parts of the lab went well.

**Graded Quiz points:**  
A 10 point graded Coursera Assignment quiz is given to assess your understanding
from this lab experience. It is **mandatory**, and you need **80%** to pass.

| Quiz | Points | Comes from |
|---|---:|---|
| Concept Check quiz | 10 | Auto-graded Coursera Assignment |

> **Rounding is fine.** Every numeric check allows 2% relative tolerance (the
> fitted slope, which is noisier by nature, allows more). Transfer the values as
> the lab produced them and you will be well inside it.

> **Do not skip the Concept Check quiz.** It lives in the Coursera **Graded
> Assignment** section, is auto-graded as its own item, and is worth **10 graded
> points** — the only points for this lab that count toward your course grade.
> It is mandatory and requires 80% to pass.


## Use your own measurements

These 40 points are ungraded, so there is nothing to gain by editing numbers —
the value of this notebook is the feedback. Use measurements from **your own**
Colab run of Lab 4.

Be aware that the checks in Part B are *cross-checks*: they recompute each
derived quantity from your own raw inputs and require the two to agree. The
bandwidth must follow from the bytes and the time; the slope must follow from the
sweep points; the KV byte count must follow from your model config. Numbers that
were not produced together by a real run generally fail several of these at once.
Transcription and rounding are fine; invented values are not.


---
## Your results  &nbsp;·&nbsp; *start here*

Paste the **entire contents** of `lab4_results.json` from your Colab run into the
next cell, between the triple quotes — from the opening `{` through the closing `}`.

Keep the raw-string wrapper (the `r` and the triple quotes) exactly as it is: it
means backslashes and quotes inside your JSON cannot break the cell.

```python
RESULTS_JSON = r"""
paste the contents of lab4_results.json here
"""
```

> **This cell only stores your text — it does not check it yet.** After pasting,
> use **Kernel → Restart & Run All**: the Setup cell below loads the helpers, and
> every check in Parts A–C runs against what you pasted here.

> **If you cannot paste**, upload `lab4_results.json` next to this notebook
> instead (Coursera Labs: the **upload** button in the file browser) and leave the
> placeholder text alone. The loader checks the pasted text first, then falls back
> to the file.


In [ ]:
# Paste the contents of lab4_results.json between the triple quotes.
# (Leave the surrounding raw-string triple quotes exactly as they are.)

# INSTRUCTOR NOTE (inside the solution block, never shown to students):
# Every value below is a real measurement from an end-to-end execution of
# Lab4_Profiling_Memory_Access_SOLUTION.ipynb on a Colab free-tier T4
# (torch 2.11.0+cu130, distilgpt2 in fp16). Nothing here is synthesised.
#
# It is the reference the Parts A-C checks were tuned against: 40/40.
# If you re-run the solution and the numbers move, paste the new JSON here --
# the checks are self-consistency relations, not absolute magnitudes, so any
# genuine run passes. A student whose model loads in fp32 measures roughly
# twice these memory figures and still scores 40/40, which is intended.

# your code here
raise NotImplementedError
RESULTS_JSON = r"""
paste the contents of lab4_results.json here
"""


---
## Setup

The next cell defines the loader and the comparison helpers used by every
check below. It is read-only — just run it. It imports only the Python standard
library: nothing here needs `torch`, a GPU, or the network.

In [ ]:
# === Grading helpers (read-only) =======================================
# Standard library only: nothing here needs torch, a GPU, or the network.
import json, ast, os, re

TOL       = 0.02    # 2% relative tolerance on derived quantities
SLOPE_TOL = 0.10    # the fitted slope is noisier by nature; allow 10%

# distilgpt2 is fixed by the lab. These are checked against the config the
# student exported, so a different model is caught rather than silently accepted.
EXP_N_LAYER, EXP_N_HEAD, EXP_N_EMBD = 6, 12, 768
EXP_HEAD_DIM = EXP_N_EMBD // EXP_N_HEAD          # 64
EXP_KV_BYTES_PER_TOKEN = 2 * EXP_N_LAYER * EXP_N_HEAD * EXP_HEAD_DIM * 2   # 18,432

# Top-level keys the Part 8 export writes. A missing one means the lab did not
# finish, or the notebook is an older version.
CORE_KEYS = ("model_config", "peak_bytes", "top_ops", "copy_bw",
             "peak_sweep", "kv", "kv_variants", "scratchpad",
             "bottlenecks", "landscape_row_w4")

REQUIRED_LS_KEYS = {
    "accelerator", "memory_hierarchy", "peak_alloc_MB",
    "kv_bytes_per_token", "measured_bw_GBps", "top_mem_operator",
}

# The Part 6 controlled vocabularies, identical to the lab notebook's.
BOTTLENECK_WHY = {
    "low_arithmetic_intensity", "large_intermediate_tensor",
    "weight_reread_per_token", "random_access_low_reuse", "kv_cache_growth",
}
BOTTLENECK_TARGET = {"spyre", "gpu", "comparable"}

# Plausible band for a measured device-to-device copy, from a throttled T4
# (~200 GB/s) to an A100 (~1.5 TB/s), widened generously at both ends. A copy
# never reaches the datasheet peak, so the upper bound is not a spec figure.
BW_BAND = (10.0, 3000.0)

# --- score bookkeeping (for the courtesy readout near the end) ----------
# Each graded cell calls record(...) as its LAST line, after its asserts. A
# cell that raises never reaches that line, so "absent" means "failed".
_SCORE = {}

def record(gid):
    """Mark a graded cell as passed. Called at the end of each graded cell."""
    _SCORE[gid] = True

# Every autograded cell, with its part, points and label. The tally walks this,
# so it must stay in step with the cells' nbgrader metadata (a build-time test
# enforces that, and the assert below enforces the total).
MANIFEST = [
    ('a_units_peak_bytes',          'A',  2, 'peak_bytes is raw bytes, not megabytes'),
    ('a_model_identity',            'A',  3, 'model config is distilgpt2 and self-consistent'),
    ('a_top_ops_shape',             'A',  2, 'top_ops rows are (op, mem_MB, time_ms, calls)'),
    ('a_top_ops_sorted',            'A',  2, 'top_ops ranked by memory, descending'),
    ('a_landscape_keys',            'A',  2, 'landscape_row_w4 has all 6 fields, filled'),
    ('a_bottlenecks_form',          'A',  3, '3 distinct bottlenecks, valid codes, own ops'),
    ('b_kv_formula',                'B',  4, 'kv bytes/token follows from your own config'),
    ('b_kv_variants',               'B',  3, 'MHA/GQA/MQA ratios are exactly 4x and 32x'),
    ('b_copy_bytes_moved',          'B',  3, 'bytes_moved = 2 x size (read + write)'),
    ('b_copy_bandwidth',            'B',  3, 'bw_gbs follows from bytes_moved and time'),
    ('b_slope_refit',               'B',  3, 'reported slope matches a refit of your sweep'),
    ('b_landscape_mirrors',         'B',  2, 'landscape row mirrors your raw measurements'),
    ('c_sweep_monotonic',           'C',  2, 'peak memory grows with generation length'),
    ('c_bandwidth_band',            'C',  2, 'measured bandwidth is physically plausible'),
    ('c_proxy_not_bandwidth',       'C',  2, 'allocator proxy differs from real bandwidth'),
    ('c_scratchpad_sizing',         'C',  2, 'scratchpad capacity arithmetic is consistent'),
]
assert sum(p for _g, _p, p, _l in MANIFEST) == 40, "manifest must total 40 points"

CHECK_LABELS = {g: lbl for g, _p, _pt, lbl in MANIFEST}
PART_NAMES = {
    "A": "Structural invariants",
    "B": "Self-consistency",
    "C": "Physics and plausibility",
}


def rel(a, b):
    """Relative difference between a and b, safe at b == 0."""
    return abs(float(a) - float(b)) / max(abs(float(b)), 1e-12)


def fit_slope(xs, ys):
    """Least-squares slope of ys vs xs. Identical arithmetic to the lab's."""
    n  = len(xs)
    mx = sum(xs) / n
    my = sum(ys) / n
    den = sum((x - mx) ** 2 for x in xs)
    if den == 0:
        raise AssertionError(
            "every point in peak_sweep has the same new_tokens value, so no "
            "slope can be fitted. Re-run Part 5 in Colab.")
    return sum((x - mx) * (y - my) for x, y in zip(xs, ys)) / den


def num(x, field="value"):
    """Read x as a float, forgiving the ways a real export can mangle one.

    The lab writes plain JSON, but students paste through editors, so a value
    can arrive as "1271.0", "349.0 MB", "tensor(64.)" or "np.float64(64.0)"
    instead of a number. Coerce rather than fail: a unit suffix or a repr
    wrapper is a formatting accident, not a wrong answer.
    """
    if isinstance(x, bool):
        raise AssertionError(f"{field}: expected a number, got a boolean.")
    if x is None:
        raise AssertionError(
            f"{field}: is missing (None). Re-run the final cell of the Colab lab.")
    if isinstance(x, (int, float)):
        return float(x)
    if isinstance(x, str):
        t = x.strip().replace(",", "")
        # tensor(64.) / np.float64(64.0) / Decimal('64') -> 64
        m = re.fullmatch(r"[A-Za-z_][A-Za-z0-9_.]*\(\s*['\"]?([-\d.eE+]+)['\"]?\s*\)", t)
        if m:
            t = m.group(1)
        for suffix in ("bytes", "GB/s", "GBps", "MB/s", "tokens", "tok",
                       "MB", "mb", "GB", "ms", "us", "x"):
            if t.lower().endswith(suffix.lower()):
                t = t[: -len(suffix)].strip()
        try:
            return float(t)
        except ValueError:
            pass
    raise AssertionError(f"{field}: could not read {x!r} as a number.")


def _parse(text):
    text = text.strip()
    if not text:
        raise AssertionError("No results provided.")
    try:
        return json.loads(text)                 # normal path
    except Exception:
        pass
    try:
        return ast.literal_eval(text)           # tolerate a pasted Python dict
    except Exception:
        pass
    if text.startswith("{") and not text.endswith("}"):
        raise AssertionError(
            "Your pasted JSON looks incomplete. Copy from the opening '{' "
            "through the final '}'.")
    raise AssertionError(
        "Could not parse your results. Paste the exact contents of "
        "lab4_results.json. (Curly 'smart quotes' from a word processor "
        "will break JSON - paste as plain text.)")


def load_results(blob=""):
    """Pasted JSON first, then an uploaded lab4_results.json, else explain."""
    data, src = None, None
    if blob and blob.strip() and "paste the contents" not in blob:
        data, src = _parse(blob), "pasted text"
    else:
        for p in ("lab4_results.json", "submission/lab4_results.json",
                  os.path.expanduser("~/lab4_results.json")):
            if os.path.exists(p):
                with open(p) as f:
                    data, src = _parse(f.read()), p
                break
    if data is None:
        raise AssertionError(
            "No results found. Either paste the contents of lab4_results.json "
            "into RESULTS_JSON below, or upload the file next to this notebook.")
    if not isinstance(data, dict):
        raise AssertionError("Your results should be a JSON object (a dict).")

    missing = [k for k in CORE_KEYS if k not in data]
    if missing:
        raise AssertionError(
            f"Key(s) {missing} are missing from your results. Re-run Part 8 of "
            f"the Colab lab to regenerate lab4_results.json. (The current Part 8 "
            f"cell exports {len(CORE_KEYS)} graded top-level keys plus "
            f"'alloc_proxy_GBps' and 'environment'.)")

    # ---- model config -------------------------------------------------
    mc_in = dict(data["model_config"])
    for k in ("n_layer", "n_head", "n_embd", "head_dim", "n_params",
              "bytes_per_element"):
        if k not in mc_in:
            raise AssertionError(
                f"model_config is missing '{k}'. Re-run Part 8 in Colab.")
    mc = {k: num(mc_in[k], f"model_config['{k}']")
          for k in ("n_layer", "n_head", "n_embd", "head_dim", "n_params",
                    "bytes_per_element")}
    mc["model_name"] = str(mc_in.get("model_name", ""))

    # ---- Part 3 -------------------------------------------------------
    peak_bytes = num(data["peak_bytes"], "peak_bytes")

    top_ops = []
    for e in data["top_ops"]:
        if not isinstance(e, (list, tuple)) or len(e) != 4:
            raise AssertionError(
                "Each top_ops entry should be a 4-element "
                "[op, self_mem_MB, time_ms, calls] list.")
        top_ops.append((str(e[0]),
                        num(e[1], "a top_ops self_mem_MB"),
                        num(e[2], "a top_ops time_ms"),
                        num(e[3], "a top_ops call count")))
    if not top_ops:
        raise AssertionError("top_ops is empty; Part 3 produced no operators.")

    # ---- Part 4 -------------------------------------------------------
    cb_in = dict(data["copy_bw"])
    for k in ("size_mb", "bytes_moved", "median_ms", "bw_gbs"):
        if k not in cb_in:
            raise AssertionError(f"copy_bw is missing '{k}'. Re-run Part 8 in Colab.")
    copy_bw = {k: num(cb_in[k], f"copy_bw['{k}']")
               for k in ("size_mb", "bytes_moved", "median_ms", "bw_gbs")}
    if copy_bw["median_ms"] <= 0:
        raise AssertionError("copy_bw['median_ms'] must be positive.")

    alloc_proxy = num(data.get("alloc_proxy_GBps", 0.0), "alloc_proxy_GBps")

    # ---- Part 5 -------------------------------------------------------
    sweep = []
    for e in data["peak_sweep"]:
        if not isinstance(e, (list, tuple)) or len(e) != 2:
            raise AssertionError(
                "Each peak_sweep entry should be a [new_tokens, peak_MB] pair.")
        sweep.append((num(e[0], "a peak_sweep new_tokens"),
                      num(e[1], "a peak_sweep peak_MB")))
    if len(sweep) < 4:
        raise AssertionError(
            f"peak_sweep has {len(sweep)} points; Part 5 sweeps at least 4 "
            f"lengths. Re-run it in Colab.")

    kv_in = dict(data["kv"])
    for k in ("bytes_per_token", "slope_MB_per_token"):
        if k not in kv_in:
            raise AssertionError(f"kv is missing '{k}'. Re-run Part 8 in Colab.")
    kv = {
        "bytes_per_token":    num(kv_in["bytes_per_token"], "kv['bytes_per_token']"),
        "slope_MB_per_token": num(kv_in["slope_MB_per_token"], "kv['slope_MB_per_token']"),
        "analytic_MB_per_token": num(kv_in.get("analytic_MB_per_token",
                                               kv_in["bytes_per_token"] / 1e6),
                                     "kv['analytic_MB_per_token']"),
    }

    # ---- Part 5b ------------------------------------------------------
    kvv_in = dict(data["kv_variants"])
    for k in ("mha_bytes", "gqa_bytes", "mqa_bytes"):
        if k not in kvv_in:
            raise AssertionError(
                f"kv_variants is missing '{k}'. Re-run Part 8 in Colab.")
    kv_variants = {k: num(v, f"kv_variants['{k}']") for k, v in kvv_in.items()}
    if kv_variants["gqa_bytes"] <= 0 or kv_variants["mqa_bytes"] <= 0:
        raise AssertionError("kv_variants byte counts must be positive.")

    sp_in = dict(data["scratchpad"])
    for k in ("total_MB", "bytes_per_token", "max_seq_kv_only", "weights_MB"):
        if k not in sp_in:
            raise AssertionError(
                f"scratchpad is missing '{k}'. Re-run Part 8 in Colab.")
    scratchpad = {k: num(sp_in[k], f"scratchpad['{k}']")
                  for k in ("total_MB", "bytes_per_token", "max_seq_kv_only",
                            "weights_MB")}

    # ---- Part 6 / 7 ---------------------------------------------------
    bottlenecks = []
    for e in data["bottlenecks"]:
        if not isinstance(e, dict):
            raise AssertionError(
                "Each bottlenecks entry should be a dict with keys "
                "op, why, better_on.")
        bottlenecks.append({k: str(e.get(k, "")) for k in ("op", "why", "better_on")})

    return {
        "model_config": mc,
        "peak_bytes": peak_bytes,
        "peak_alloc_MB": num(data.get("peak_alloc_MB", peak_bytes / 1e6),
                             "peak_alloc_MB"),
        "top_ops": top_ops,
        "copy_bw": copy_bw,
        "alloc_proxy_GBps": alloc_proxy,
        "peak_sweep": sweep,
        "kv": kv,
        "kv_variants": kv_variants,
        "scratchpad": scratchpad,
        "bottlenecks": bottlenecks,
        "landscape_row_w4": dict(data["landscape_row_w4"]),
        "_source": src,
    }


print("Helpers loaded. Tolerance on derived values: "
      f"{TOL*100:.0f}%  (fitted slope: {SLOPE_TOL*100:.0f}%)")
print(f"Autograded cells: {len(MANIFEST)}   "
      f"Parts A-C total: {sum(p for _g, _p, p, _l in MANIFEST)} points"
      f"   (fully autograded)")


In [ ]:
# === Load and echo your results (read-only) ============================
# Every check below calls load_results() itself, so one bad field can
# never cascade into unrelated failures.
_SCORE.clear()   # fresh top-to-bottom run: forget any earlier cell outcomes

_r  = load_results(RESULTS_JSON)
_mc = _r["model_config"]
_cb = _r["copy_bw"]

print(f"Loaded from: {_r['_source']}\n")
print(f"  model               {_mc['model_name'] or '(unnamed)'}")
print(f"    layers x heads x head_dim   {_mc['n_layer']:.0f} x "
      f"{_mc['n_head']:.0f} x {_mc['head_dim']:.0f}")
print(f"    parameters                  {_mc['n_params']/1e6:.1f} M\n")
print(f"  peak_bytes          {_r['peak_bytes']:>15,.0f} bytes "
      f"({_r['peak_bytes']/1e6:.1f} MB)")
print(f"  peak_alloc_MB       {_r['peak_alloc_MB']:>15.1f} MB\n")
print("  Top operators by memory:")
for _n, _m, _t, _c in _r["top_ops"][:5]:
    print(f"    {_n[:34]:34s} {_m:9.2f} MB  {_t:9.2f} ms  x{_c:.0f}")
print()
print(f"  copy bandwidth      {_cb['bw_gbs']:>15.1f} GB/s "
      f"({_cb['bytes_moved']:,.0f} bytes in {_cb['median_ms']:.3f} ms)")
print(f"  allocator proxy     {_r['alloc_proxy_GBps']:>15.2f} GB/s "
      f"<- NOT bandwidth\n")
print("  Peak-memory sweep:")
for _n, _p in _r["peak_sweep"]:
    print(f"    {_n:6.0f} new tokens  ->  {_p:9.2f} MB")
_refit = fit_slope([p[0] for p in _r["peak_sweep"]],
                   [p[1] for p in _r["peak_sweep"]])
print(f"\n  KV bytes/token      {_r['kv']['bytes_per_token']:>15,.0f} bytes")
print(f"  slope (reported)    {_r['kv']['slope_MB_per_token']:>15.6f} MB/token")
print(f"  slope (refitted)    {_refit:>15.6f} MB/token "
      f"<- from your sweep; Part B checks these agree\n")
print(f"  scratchpad          {_r['scratchpad']['total_MB']:.0f} MB holds "
      f"{_r['scratchpad']['max_seq_kv_only']:,.0f} tokens of KV")
print(f"  weights             {_r['scratchpad']['weights_MB']:.1f} MB\n")
print("  Bottleneck annotation:")
for _b in _r["bottlenecks"]:
    print(f"    {_b['op'][:26]:26s} {_b['why']:26s} -> {_b['better_on']}")
print(f"\n  Accelerator: {_r['landscape_row_w4'].get('accelerator')}")
print("\nCheck these against your Colab output before continuing.")


---
# Part A — Structural invariants (14 ungraded Self-Assessment points)

These hold for **any** correct run on **any** GPU. They check that your results
are complete, in the right units, and describe the model the lab actually used.

In [ ]:
# A1 (2 pts): peak_bytes must be raw BYTES from torch.cuda.max_memory_allocated().
_r = load_results(RESULTS_JSON)
_pb = _r["peak_bytes"]
assert _pb > 1e6, (
    f"peak_bytes = {_pb:g} looks like megabytes, not bytes. Use the raw value "
    f"from torch.cuda.max_memory_allocated() (expect a few times 1e8 for this lab).")
assert _pb < 1e12, f"peak_bytes = {_pb:g} is implausibly large for distilgpt2."

# peak_alloc_MB is the rounded display value; it must round-trip.
_e = rel(_r["peak_alloc_MB"], _pb / 1e6)
assert _e <= 0.01, (
    f"peak_alloc_MB = {_r['peak_alloc_MB']:.1f} but peak_bytes / 1e6 = "
    f"{_pb/1e6:.1f} (off by {_e*100:.1f}%). They describe the same measurement.")
print(f"[OK] peak_bytes = {_pb:,.0f} bytes ({_pb/1e6:.1f} MB), units consistent")
record("a_units_peak_bytes")


In [ ]:
# A2 (3 pts): the exported config must describe distilgpt2 and be self-consistent.
_r = load_results(RESULTS_JSON)
_mc = _r["model_config"]

# head_dim is n_embd / n_head by definition -- a derived field, so check it.
_hd_expected = _mc["n_embd"] / _mc["n_head"]
assert abs(_mc["head_dim"] - _hd_expected) < 1e-6, (
    f"head_dim = {_mc['head_dim']:.0f} but n_embd / n_head = "
    f"{_hd_expected:.1f}. head_dim is defined as n_embd // n_head.")

assert (_mc["n_layer"], _mc["n_head"], _mc["n_embd"]) == (
    float(EXP_N_LAYER), float(EXP_N_HEAD), float(EXP_N_EMBD)), (
    f"this lab profiles distilgpt2 ({EXP_N_LAYER} layers, {EXP_N_HEAD} heads, "
    f"n_embd {EXP_N_EMBD}), but your config says {_mc['n_layer']:.0f} / "
    f"{_mc['n_head']:.0f} / {_mc['n_embd']:.0f}. Read them from model.config.")

# distilgpt2 is ~82 M parameters. Wide band, but a different model is caught.
assert 6.0e7 <= _mc["n_params"] <= 1.0e8, (
    f"your config implies {_mc['n_params']/1e6:.1f} M parameters, but "
    f"distilgpt2 has about 82 M.")

assert _mc["bytes_per_element"] == 2.0, (
    f"bytes_per_element = {_mc['bytes_per_element']:.0f}; the KV-cache figures "
    f"in this lab are at FP16, which is 2 bytes.")

print(f"[OK] distilgpt2 confirmed: {_mc['n_layer']:.0f} layers, "
      f"{_mc['n_head']:.0f} heads, head_dim {_mc['head_dim']:.0f}, "
      f"{_mc['n_params']/1e6:.1f} M params")
record("a_model_identity")


In [ ]:
# A3 (2 pts): top_ops must be well-formed rows from the Part 3 table.
_r = load_results(RESULTS_JSON)
_ops = _r["top_ops"]
assert 1 <= len(_ops) <= 8, (
    f"top_ops has {len(_ops)} rows; Part 3 asks for the top k=8, so expect "
    f"1-8. Re-run Part 3 and Part 8 in Colab.")
for _i, (_n, _m, _t, _c) in enumerate(_ops):
    assert _n and _n.strip(), f"top_ops[{_i}] has an empty operator name."
    assert _c >= 1, (
        f"top_ops[{_i}] ('{_n}') reports {_c:.0f} calls; a profiled operator "
        f"runs at least once.")
    assert _t >= 0, f"top_ops[{_i}] ('{_n}') has negative time: {_t}."
# Every row in the table was filtered to non-zero memory.
assert all(abs(_m) > 0 for _n, _m, _t, _c in _ops), (
    "a top_ops row reports zero memory. Part 3 filters those out: "
    "df[df['self_mem_MB'].abs() > 0].")
print(f"[OK] {len(_ops)} well-formed operator rows; dominant: {_ops[0][0]}")
record("a_top_ops_shape")


In [ ]:
# A4 (2 pts): top_ops must be ranked by memory, descending.
# The grader -- and your landscape row -- read top_ops[0] as the dominant
# memory operator, so the ordering carries meaning.
_r = load_results(RESULTS_JSON)
_mem = [_m for _n, _m, _t, _c in _r["top_ops"]]
assert _mem == sorted(_mem, reverse=True), (
    f"top_ops is not sorted by self_mem_MB descending: {[round(m,2) for m in _mem]}. "
    f"Part 3 sorts with .sort_values('self_mem_MB', ascending=False).")
print(f"[OK] ranked by memory: {_mem[0]:.2f} MB down to {_mem[-1]:.2f} MB")
record("a_top_ops_sorted")


In [ ]:
# A5 (2 pts): landscape_row_w4 must have exactly the 6 required fields, filled.
_r = load_results(RESULTS_JSON)
_ls = _r["landscape_row_w4"]
_got = set(_ls)
assert _got == REQUIRED_LS_KEYS, (
    f"landscape_row_w4 should have exactly these keys: "
    f"{sorted(REQUIRED_LS_KEYS)}. Missing: {sorted(REQUIRED_LS_KEYS - _got)}; "
    f"unexpected: {sorted(_got - REQUIRED_LS_KEYS)}.")
_empty = [k for k, v in _ls.items()
          if v is None or (isinstance(v, str) and not v.strip())]
assert not _empty, f"these landscape_row_w4 fields are still empty: {_empty}"
# The two text fields should be real descriptions, not placeholders.
for _k in ("accelerator", "memory_hierarchy"):
    assert len(str(_ls[_k]).strip()) >= 3, (
        f"landscape_row_w4['{_k}'] = {_ls[_k]!r} is too short to be a "
        f"description.")
print(f"[OK] all 6 landscape fields present and filled")
print(f"     {_ls['accelerator']}  |  {_ls['memory_hierarchy']}")
record("a_landscape_keys")


In [ ]:
# A6 (3 pts): the Part 6 annotation must be well-formed.
#
# We never grade WHICH codes you chose -- there is no single right assignment.
# We grade that you annotated three distinct operators from your OWN profile
# using the published vocabularies.
_r = load_results(RESULTS_JSON)
_bs = _r["bottlenecks"]
_own = {_n for _n, _m, _t, _c in _r["top_ops"]}

assert len(_bs) == 3, (
    f"bottlenecks should have exactly 3 entries, got {len(_bs)}.")

_seen = set()
for _i, _b in enumerate(_bs):
    assert _b["op"], f"bottlenecks[{_i}] has an empty 'op'."
    assert _b["op"] in _own, (
        f"bottlenecks[{_i}] names '{_b['op']}', which is not in your own "
        f"top_ops. Annotate operators your run actually profiled: "
        f"{sorted(_own)[:5]}.")
    assert _b["why"] in BOTTLENECK_WHY, (
        f"bottlenecks[{_i}]: why = {_b['why']!r} is not one of "
        f"{sorted(BOTTLENECK_WHY)}.")
    assert _b["better_on"] in BOTTLENECK_TARGET, (
        f"bottlenecks[{_i}]: better_on = {_b['better_on']!r} is not one of "
        f"{sorted(BOTTLENECK_TARGET)}.")
    assert _b["op"] not in _seen, (
        f"'{_b['op']}' is annotated twice; pick three distinct operators.")
    _seen.add(_b["op"])

print("[OK] 3 distinct operators, all from your own profile, all codes valid:")
for _b in _bs:
    print(f"     {_b['op'][:26]:26s} {_b['why']:26s} -> {_b['better_on']}")
record("a_bottlenecks_form")


---
# Part B — Self-consistency (18 ungraded Self-Assessment points)

Each check **recomputes** one of your derived values from your own raw
measurements and requires the two to agree. This is the heart of the
self-assessment: the relationships hold for any genuine run, and they are what
break when a number is edited by hand.

In [ ]:
# B1 (4 pts): the KV byte count must follow from YOUR OWN model config.
#
#   kv_bytes_per_token = 2 (K and V) x n_layer x n_head x head_dim x bytes
#
# Recomputed from the config you exported, not assumed. This is the lab's
# central formula, so it carries the most weight of any single check.
_r = load_results(RESULTS_JSON)
_mc = _r["model_config"]
_reported = _r["kv"]["bytes_per_token"]

_expected = (2 * _mc["n_layer"] * _mc["n_head"] * _mc["head_dim"]
             * _mc["bytes_per_element"])

assert abs(_reported - _expected) < 1e-6, (
    f"kv.bytes_per_token = {_reported:,.0f} but 2 x n_layer x n_head x "
    f"head_dim x bytes = 2 x {_mc['n_layer']:.0f} x {_mc['n_head']:.0f} x "
    f"{_mc['head_dim']:.0f} x {_mc['bytes_per_element']:.0f} = "
    f"{_expected:,.0f}. Common slips: dropping the leading 2 (K *and* V), or "
    f"using 4 bytes/element instead of 2.")

# For distilgpt2 that is exactly 18,432 -- a fixed number on any hardware.
assert abs(_reported - EXP_KV_BYTES_PER_TOKEN) < 1e-6, (
    f"for distilgpt2 the KV cache is exactly {EXP_KV_BYTES_PER_TOKEN:,} "
    f"bytes/token; you reported {_reported:,.0f}.")

# The MB figure is the same quantity, divided by 1e6.
_e = rel(_r["kv"]["analytic_MB_per_token"], _reported / 1e6)
assert _e <= TOL, (
    f"kv.analytic_MB_per_token = {_r['kv']['analytic_MB_per_token']:.6f} but "
    f"bytes_per_token / 1e6 = {_reported/1e6:.6f} (off by {_e*100:.1f}%).")

print(f"[OK] KV = {_reported:,.0f} bytes/token, recomputed from your own config")
print(f"     2 x {_mc['n_layer']:.0f} layers x {_mc['n_head']:.0f} heads x "
      f"{_mc['head_dim']:.0f} head_dim x {_mc['bytes_per_element']:.0f} bytes")
record("b_kv_formula")


In [ ]:
# B2 (3 pts): the MHA / GQA / MQA ratios are exact arithmetic.
#
# The cache scales with n_kv_heads and nothing else, so 32 vs 8 vs 1 KV heads
# gives exactly 4x and 32x. No measurement, no tolerance needed -- these are
# identical on every machine, which is why they are worth grading.
_r = load_results(RESULTS_JSON)
_v = _r["kv_variants"]
_mha, _gqa, _mqa = _v["mha_bytes"], _v["gqa_bytes"], _v["mqa_bytes"]

_r1 = _mha / _gqa
_r2 = _mha / _mqa
assert abs(_r1 - 4.0) < 0.01, (
    f"MHA / GQA = {_r1:.3f}, expected exactly 4.0 (32 KV heads vs 8). Only "
    f"n_kv_heads changes between the two calls to kv_cache_bytes().")
assert abs(_r2 - 32.0) < 0.01, (
    f"MHA / MQA = {_r2:.3f}, expected exactly 32.0 (32 KV heads vs 1).")

# And the absolute MHA figure must match the reference configuration.
if all(k in _v for k in ("ref_layers", "ref_heads", "ref_head_dim", "ref_seq_len")):
    _exp_mha = (2 * _v["ref_layers"] * _v["ref_heads"] * _v["ref_head_dim"]
                * _v["ref_seq_len"] * 2)
    _e = rel(_mha, _exp_mha)
    assert _e <= TOL, (
        f"mha_bytes = {_mha:,.0f} but 2 x {_v['ref_layers']:.0f} x "
        f"{_v['ref_heads']:.0f} x {_v['ref_head_dim']:.0f} x "
        f"{_v['ref_seq_len']:.0f} x 2 = {_exp_mha:,.0f} (off by {_e*100:.1f}%).")

print(f"[OK] MHA {_mha/1e6:.1f} MB / GQA {_gqa/1e6:.1f} MB / MQA {_mqa/1e6:.1f} MB")
print(f"     ratios exactly {_r1:.0f}x and {_r2:.0f}x = n_heads / n_kv_heads")
record("b_kv_variants")


In [ ]:
# B3 (3 pts): a copy reads AND writes, so traffic is 2x the tensor size.
#
# This is the single most-missed point in Part 4. Dropping the factor of 2
# halves the reported bandwidth, and nothing in the printed output looks wrong.
_r = load_results(RESULTS_JSON)
_cb = _r["copy_bw"]
_expected = 2 * _cb["size_mb"] * 1e6

_e = rel(_cb["bytes_moved"], _expected)
if rel(_cb["bytes_moved"], _cb["size_mb"] * 1e6) <= TOL:
    raise AssertionError(
        f"bytes_moved = {_cb['bytes_moved']:,.0f} is exactly the tensor size "
        f"({_cb['size_mb']:.0f} MB). A copy READS the source and WRITES the "
        f"destination, so the traffic is 2 x {_cb['size_mb']:.0f} MB = "
        f"{_expected:,.0f} bytes. That factor of 2 is the point of the exercise.")
assert _e <= TOL, (
    f"bytes_moved = {_cb['bytes_moved']:,.0f} but a {_cb['size_mb']:.0f} MB "
    f"copy moves 2 x {_cb['size_mb']:.0f} MB = {_expected:,.0f} bytes "
    f"(off by {_e*100:.1f}%).")

print(f"[OK] bytes_moved = {_cb['bytes_moved']:,.0f} = 2 x "
      f"{_cb['size_mb']:.0f} MB (read + write)")
record("b_copy_bytes_moved")


In [ ]:
# B4 (3 pts): bandwidth must follow from the bytes moved and the time taken.
_r = load_results(RESULTS_JSON)
_cb = _r["copy_bw"]
_expected = (_cb["bytes_moved"] / 1e9) / (_cb["median_ms"] / 1e3)

_e = rel(_cb["bw_gbs"], _expected)
assert _e <= TOL, (
    f"bw_gbs = {_cb['bw_gbs']:.1f} GB/s but bytes_moved / time = "
    f"({_cb['bytes_moved']:,.0f} / 1e9) / ({_cb['median_ms']:.3f} / 1e3) = "
    f"{_expected:.1f} GB/s, off by {_e*100:.1f}%. These must agree: the "
    f"bandwidth is nothing but that division.")

# The landscape row quotes the same number.
_ls_bw = num(_r["landscape_row_w4"]["measured_bw_GBps"],
             "landscape_row_w4['measured_bw_GBps']")
_e2 = rel(_ls_bw, _cb["bw_gbs"])
assert _e2 <= 0.02, (
    f"landscape_row_w4['measured_bw_GBps'] = {_ls_bw:.1f} but your Part 4.2 "
    f"measurement was {_cb['bw_gbs']:.1f} GB/s (off by {_e2*100:.1f}%). The "
    f"row should quote the MEASURED bandwidth, not the 4.1 allocator proxy "
    f"({_r['alloc_proxy_GBps']:.2f} GB/s).")

print(f"[OK] {_cb['bw_gbs']:.1f} GB/s follows from {_cb['bytes_moved']:,.0f} "
      f"bytes in {_cb['median_ms']:.3f} ms")
record("b_copy_bandwidth")


In [ ]:
# B5 (3 pts): the reported slope must match a refit of your own sweep points.
#
# The grader runs the SAME least-squares arithmetic on your (new_tokens, peak_MB)
# pairs. Note what this does NOT do: it never compares your slope to the analytic
# KV figure. The measured slope is legitimately higher (attention scores and
# allocator block rounding also scale with length), and how much higher is
# hardware- and allocator-specific. What must hold is that your slope came from
# your sweep.
_r = load_results(RESULTS_JSON)
_sweep = _r["peak_sweep"]
_refit = fit_slope([p[0] for p in _sweep], [p[1] for p in _sweep])
_reported = _r["kv"]["slope_MB_per_token"]

assert _refit != 0, "your sweep is perfectly flat; no slope can be fitted."
_e = rel(_reported, _refit)
assert _e <= SLOPE_TOL, (
    f"kv.slope_MB_per_token = {_reported:.6f} MB/token, but refitting a "
    f"least-squares line through your own peak_sweep gives {_refit:.6f} "
    f"(off by {_e*100:.1f}%, tolerance {SLOPE_TOL*100:.0f}%). The slope must "
    f"come from the sweep you exported. If you changed the sweep lengths after "
    f"fitting, re-run Part 5 and Part 8 together.")

print(f"[OK] slope {_reported:.6f} MB/token matches the refit {_refit:.6f} "
      f"from your {len(_sweep)} sweep points")
print(f"     (analytic KV is {_r['kv']['bytes_per_token']/1e6:.6f} MB/token; "
      f"measured is {_refit/max(_r['kv']['bytes_per_token']/1e6,1e-12):.2f}x that "
      f"-- not graded, correctly so)")
record("b_slope_refit")


In [ ]:
# B6 (2 pts): the landscape row must mirror the raw measurements behind it.
_r = load_results(RESULTS_JSON)
_ls = _r["landscape_row_w4"]

# peak_alloc_MB <-> peak_bytes
_v = num(_ls["peak_alloc_MB"], "landscape_row_w4['peak_alloc_MB']")
_e = rel(_v, _r["peak_bytes"] / 1e6)
assert _e <= 0.01, (
    f"landscape peak_alloc_MB = {_v:.1f} but peak_bytes / 1e6 = "
    f"{_r['peak_bytes']/1e6:.1f} (off by {_e*100:.1f}%).")

# kv_bytes_per_token <-> kv
_v = num(_ls["kv_bytes_per_token"], "landscape_row_w4['kv_bytes_per_token']")
assert abs(_v - _r["kv"]["bytes_per_token"]) < 1e-6, (
    f"landscape kv_bytes_per_token = {_v:,.0f} but Part 5 computed "
    f"{_r['kv']['bytes_per_token']:,.0f}.")

# top_mem_operator <-> top_ops[0][0]
_dom = str(_ls["top_mem_operator"]).strip()
_top = _r["top_ops"][0][0]
assert _dom == _top, (
    f"landscape top_mem_operator = '{_dom}' but the highest-memory row in your "
    f"top_ops is '{_top}'. The dominant operator is top_ops[0].")

print(f"[OK] landscape row mirrors your measurements: {_v:,.0f} B/token, "
      f"{_r['peak_bytes']/1e6:.1f} MB peak, dominant '{_dom}'")
record("b_landscape_mirrors")


---
# Part C — Physics and plausibility (8 ungraded Self-Assessment points)

Wide sanity bands. These confirm your measurements make physical sense and that
you drew the lab's central distinction — allocator activity is not bandwidth. The
bands are deliberately generous to accommodate whatever hardware Colab assigned
you and however contended the VM was.

In [ ]:
# C1 (2 pts): peak memory must grow as the generation gets longer.
#
# That growth IS the KV cache. A flat or falling sweep means the peak counter
# was not reset between lengths, or the sweep did not reach far enough for the
# cache to clear the noise floor.
_r = load_results(RESULTS_JSON)
_sweep = sorted(_r["peak_sweep"], key=lambda p: p[0])
_xs = [p[0] for p in _sweep]
_ys = [p[1] for p in _sweep]

assert len(set(_xs)) == len(_xs), (
    f"peak_sweep repeats a generation length: {[int(x) for x in _xs]}.")
assert all(y > 0 for y in _ys), (
    "every peak_MB in the sweep must be positive. On a CPU-only run the sweep "
    "returns NaN -- this lab needs a GPU runtime.")

# Allow 1 MB of slack per step for allocator noise, but the trend must be up.
_drops = [(int(a), round(ya, 2), int(b), round(yb, 2))
          for (a, ya), (b, yb) in zip(_sweep, _sweep[1:]) if yb < ya - 1.0]
assert not _drops, (
    f"peak memory fell as the generation lengthened: {_drops}. Call "
    f"torch.cuda.reset_peak_memory_stats() before each length in "
    f"peak_for_length().")

_slope = fit_slope(_xs, _ys)
assert _slope > 0, (
    f"the fitted slope is {_slope:.6f} MB/token. Peak memory must grow with "
    f"sequence length. If your sweep stopped at 128 tokens, the KV cache "
    f"(~2 MB there) is inside the noise -- the lab sweeps to 512 for exactly "
    f"this reason.")

print(f"[OK] sweep rises from {_ys[0]:.1f} MB at {_xs[0]:.0f} tokens to "
      f"{_ys[-1]:.1f} MB at {_xs[-1]:.0f} ({_ys[-1]-_ys[0]:+.1f} MB)")
record("c_sweep_monotonic")


In [ ]:
# C2 (2 pts): the measured bandwidth must be physically plausible.
_r = load_results(RESULTS_JSON)
_bw = _r["copy_bw"]["bw_gbs"]
_lo, _hi = BW_BAND
assert _lo <= _bw <= _hi, (
    f"measured bandwidth {_bw:.1f} GB/s is outside the plausible range "
    f"{_lo:.0f}-{_hi:.0f} GB/s for a device-to-device copy. If it is very low, "
    f"check that the tensors were on the GPU; if impossibly high, check that "
    f"you synchronized on BOTH sides of the timed region -- an unsynchronized "
    f"CUDA timing measures the launch, not the copy.")

# A real copy never reaches the datasheet peak. If a student reports a number
# far above every current GPU's spec, something is wrong with the timing.
print(f"[OK] {_bw:.1f} GB/s is plausible for a measured copy")
print(f"     (a T4's 320 GB/s spec peak is a ceiling, not a target -- real "
      f"copies land below it)")
record("c_bandwidth_band")


In [ ]:
# C3 (2 pts): the allocator proxy must NOT look like real bandwidth.
#
# This is Part 4's whole lesson, and the one conceptual check in Part C. The
# profiler's net-allocation-per-second and a measured copy bandwidth are
# different quantities, and on real hardware they are far apart.
_r = load_results(RESULTS_JSON)
_proxy = _r["alloc_proxy_GBps"]
_bw = _r["copy_bw"]["bw_gbs"]

assert _proxy == _proxy, (          # NaN check
    "alloc_proxy_GBps is NaN. Part 4.1 should skip operators with zero "
    "recorded time before taking the top row.")
assert _proxy >= 0, (
    f"alloc_proxy_GBps = {_proxy:.2f}; Part 4.1 takes the absolute value of "
    f"net allocation, so the proxy is non-negative.")
assert _proxy < _bw, (
    f"the allocator proxy ({_proxy:.2f} GB/s) is at or above your measured "
    f"bandwidth ({_bw:.1f} GB/s). The proxy counts NET ALLOCATION per unit "
    f"time, not bytes transferred -- it should come out far below a real copy's "
    f"bandwidth. Recheck Part 4.1: did you divide by time in seconds, and take "
    f"the row from the profiler table rather than the copy benchmark?")

_factor = _bw / max(_proxy, 1e-9)
print(f"[OK] proxy {_proxy:.2f} GB/s vs measured {_bw:.1f} GB/s "
      f"({_factor:.0f}x apart)")
print(f"     allocator activity is not bandwidth -- that is the Part 4 lesson")
record("c_proxy_not_bandwidth")


In [ ]:
# C4 (2 pts): the scratchpad capacity arithmetic must be consistent.
_r = load_results(RESULTS_JSON)
_sp = _r["scratchpad"]

# bytes_per_token here is the same quantity as kv.bytes_per_token.
assert abs(_sp["bytes_per_token"] - _r["kv"]["bytes_per_token"]) < 1e-6, (
    f"scratchpad.bytes_per_token = {_sp['bytes_per_token']:,.0f} but "
    f"kv.bytes_per_token = {_r['kv']['bytes_per_token']:,.0f}. They are the "
    f"same number.")

# max_seq_kv_only = budget / bytes-per-token, floored.
_expected = int(_sp["total_MB"] * 1e6 / _sp["bytes_per_token"])
assert abs(_sp["max_seq_kv_only"] - _expected) <= 1, (
    f"max_seq_kv_only = {_sp['max_seq_kv_only']:,.0f} but "
    f"{_sp['total_MB']:.0f} MB / {_sp['bytes_per_token']:,.0f} bytes/token = "
    f"{_expected:,}. This is the capacity question the Week 4 lecture poses: "
    f"how much context fits on-chip.")

assert _sp["total_MB"] > 0 and _sp["weights_MB"] > 0, (
    "scratchpad total_MB and weights_MB must both be positive.")

# distilgpt2 at FP16 is ~164 MB, which genuinely exceeds a ~64 MB scratchpad.
# That inequality is the setup for the placement problem, so confirm it holds.
assert _sp["weights_MB"] > _sp["total_MB"], (
    f"your numbers say the weights ({_sp['weights_MB']:.1f} MB) fit inside the "
    f"{_sp['total_MB']:.0f} MB scratchpad. distilgpt2 at FP16 is about 164 MB, "
    f"which does not -- and that mismatch is what forces the compiler to tile. "
    f"Check n_params and bytes_per_element.")

print(f"[OK] {_sp['total_MB']:.0f} MB scratchpad holds "
      f"{_sp['max_seq_kv_only']:,.0f} tokens of KV cache")
print(f"     weights {_sp['weights_MB']:.1f} MB exceed it -- hence tiling, "
      f"hence the placement problem")
record("c_scratchpad_sizing")


In [ ]:
# === Your self-assessment score (read-only readout -- nothing here is graded) ===
# Run this after running every cell above (Kernel -> Restart & Run All).
# It summarizes the checks above so you can see which parts of the lab went
# well. These points are ungraded and do not affect your course grade.
try:
    _rows, _earned, _possible = [], {}, {}
    for _gid, _part, _pts, _lbl in MANIFEST:
        _ok = _gid in _SCORE
        _earned[_part]   = _earned.get(_part, 0)   + (_pts if _ok else 0)
        _possible[_part] = _possible.get(_part, 0) + _pts
        _rows.append((_part, _gid, _pts, _ok))

    _parts  = [p for p in ("A", "B", "C") if p in _possible]
    _auto_e = sum(_earned[p]   for p in _parts)
    _auto_p = sum(_possible[p] for p in _parts)

    print("=" * 68)
    print("  YOUR SELF-ASSESSMENT SCORE  (40 ungraded points)")
    print("=" * 68)
    for _p in _parts:
        _bar = "#" * int(round(18 * _earned[_p] / max(_possible[_p], 1)))
        print(f"  {_p} - {PART_NAMES[_p]:26s} {_earned[_p]:3d} / {_possible[_p]:<3d} {_bar}")
    print(f"  {'-' * 64}")
    print(f"  TOTAL                          {_auto_e:3d} / {_auto_p:<3d}")

    # Per-check detail: every graded cell, what it checks, and the points it
    # earned. This is the full breakdown behind the part totals above.
    print("=" * 68)
    print("  DETAIL - every check, in order")
    print("=" * 68)
    _n = {}
    for _p, _g, _pt, _ok in _rows:
        _n[_p] = _n.get(_p, 0) + 1
        _tag   = f"{_p}{_n[_p]}"
        _mark  = "PASS" if _ok else "FAIL"
        _got   = _pt if _ok else 0
        _what  = CHECK_LABELS.get(_g, _g)
        print(f"  {_tag:<4s} {_mark}  {_got}/{_pt}  {_what}")
    print(f"  {'-' * 64}")
    print(f"  TOTAL      {_auto_e:2d}/{_auto_p:<2d}")

    _failed = [(p, g) for p, g, _, ok in _rows if not ok]
    if _failed:
        print("-" * 68)
        print("  Cells not yet passing - scroll up to their error messages:")
        for _p, _g in _failed:
            print(f"    [{_p}] {_g}  -  {CHECK_LABELS.get(_g, '')}")
        print("  Each is independent: fixing one cannot cost you another.")
    else:
        print("-" * 68)
        print("  Every autograded cell passed.")

    print("=" * 68)
    print("  Wrapping up")
    print("  1. Kernel -> Restart & Run All, then re-read this table. If you")
    print("     ran cells out of order, the numbers above can be stale.")
    print("  2. These 40 points are UNGRADED self-assessment feedback, auto-")
    print("     checked from your pasted JSON. No write-up, no human review,")
    print("     and nothing here affects your course grade.")
    print("  3. The GRADED item is the Concept Check quiz: a separate auto-")
    print("     graded Coursera Assignment worth 10 points. It is mandatory")
    print("     and needs 80% to pass. Do not skip it.")
    print("=" * 68)
except Exception as _e:
    # A courtesy readout must never be the reason a passing student panics.
    print(f"[score readout unavailable: {type(_e).__name__}: {_e}]")
    print("Your check cells above are unaffected - each prints [OK] when it passes.")


---
## Wrapping up

1. **Run every cell top to bottom** (Kernel → Restart & Run All). Each check
   prints `[OK]` when it passes, and the **score table** just above reports your
   40-point self-assessment total.
2. Any cell that raises an error scores 0 **for that cell only** — the checks are
   independent, so fix what you can; the rest of your self-assessment still counts.
3. Confirm the echo table under your pasted JSON matches your Colab output.
4. **Take the Concept Check quiz.** It is a separate auto-graded Coursera
   **Graded Assignment** worth **10 graded points**, it is mandatory, and you
   need **80%** to pass. The 40 points in this notebook are ungraded feedback;
   the quiz is what counts toward your grade.

### Troubleshooting

| Symptom | Fix |
|---|---|
| `Could not parse your results` | Paste as plain text; curly "smart quotes" break JSON. Copy from `{` through `}`. |
| `Your pasted JSON looks incomplete` | You copied only part of the file. Select all of it. |
| `Key(s) [...] are missing` | Re-run **Part 8** in Colab — it writes every key this notebook reads. If a key is still missing, re-download the student notebook. |
| `peak_bytes looks like megabytes` | Use the raw `torch.cuda.max_memory_allocated()` value, not the MB figure. |
| `bytes_moved is exactly the tensor size` | A copy reads *and* writes: traffic is `2 × size`. Put the factor of 2 back. |
| `bw_gbs does not follow from bytes_moved / time` | Recompute the bandwidth from the two raw values rather than transcribing a printed figure. |
| `slope ... does not match the refit` | You changed the sweep lengths after fitting. Re-run Part 5 and Part 8 together. |
| `peak memory fell as the generation lengthened` | Call `torch.cuda.reset_peak_memory_stats()` before each length in `peak_for_length()`. |
| `fitted slope is 0` or sweep flat | Your sweep stopped too early for the KV cache to clear the noise. The lab sweeps to 512 tokens for this reason. |
| `allocator proxy is at or above your measured bandwidth` | Part 4.1 divides net allocation by time; make sure you took it from the profiler table, not the copy benchmark. |
| `the weights fit inside the scratchpad` | distilgpt2 at FP16 is ~164 MB. Check `n_params` and `bytes_per_element`. |
| `MHA / GQA = ...` not 4.0 | Only `n_kv_heads` changes between the variants — 32, 8 and 1. |
| `this lab profiles distilgpt2` | You ran a different model. Read the config from `model.config` on the lab's `distilgpt2`. |

Your Colab notebook stays yours — nothing from it is uploaded except the values
you pasted here.

---

*Course: Dataflow Architectures for AI, Week 4 Lab, v1.0 · © 2026 Dr. Kaoutar El Maghraoui*